In [0]:
# Retrieve credentials securely from secret scope
PG_HOST     = dbutils.secrets.get(scope="pg_scope", key="pg_host")
PG_PORT     = dbutils.secrets.get(scope="pg_scope", key="pg_port")
PG_USER     = dbutils.secrets.get(scope="pg_scope", key="pg_user")
PG_PASSWORD = dbutils.secrets.get(scope="pg_scope", key="pg_password")
PG_DATABASE = "defaultdb"
display(PG_HOST, PG_PORT, PG_USER, PG_PASSWORD)

pg_options = {
    "host":     PG_HOST,
    "port":     PG_PORT,
    "database": PG_DATABASE,
    "user":     PG_USER,
    "password": PG_PASSWORD
}

# Reusable read/write functions
def pg_read(table_name):
    return spark.read \
        .format("postgresql") \
        .options(**pg_options) \
        .option("dbtable", table_name) \
        .load()

def pg_write(df, table_name, mode="append"):
    df.write \
        .format("postgresql") \
        .options(**pg_options) \
        .option("dbtable", table_name) \
        .mode(mode) \
        .save()

In [0]:
df = pg_read("spre_common.user")
display(df)

In [0]:
from pyspark.sql.functions import when, col, current_timestamp

df_updated = df.withColumn(
    "is_active",
    when(col("is_active") == True, "Y").otherwise("N")
).withColumn(
    "dbx_upd_ts", current_timestamp()
)

display(df_updated)

In [0]:
pg_write(df_updated, "spre_common.user_upd", mode="append")

df_tst = pg_read("spre_common.user_upd")
display(df_tst)
